# **LLM  API**



## **1.환경준비**

### (1) 구글 드라이브

* 구글 드라이브 폴더 생성
    * 새 폴더 `ai_agent`를 생성(이미 만들었다면 skip)
    * 제공 받은 파일을 업로드

* 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 라이브러리

* 필요한 라이브러리 설치

설치시 아래 메시지가 나와도 상관 없습니다.  
colab에서는 requests의 기본 버전이 2.32.4인데, 지금 2.32.5로 설치되어 혹시 오류가 날수도 있다는 경고 메시지 입니다.

    ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
    google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.

In [ ]:
!pip install langchain-openai langchain_community -q

* 라이브러리 로딩

In [ ]:
import pandas as pd
import numpy as np
import os
import openai

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

### (3) OpenAI API Key 확인

In [ ]:
# 콜랩 파일 업로드 : api_key.txt
from google.colab import files
uploaded = files.upload()

In [ ]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()


# API 키 로드 및 환경변수 설정
load_api_keys('api_key.txt')

* ⚠️ api_key가 잘 등록되었는지 확인

In [ ]:
print(os.environ['OPENAI_API_KEY'][:40])

## **2. LLM 호출 기본**

### (1) 모델 선언

In [ ]:
chat = ChatOpenAI(model = "gpt-4.1-mini")

### (2) 사용하기

In [ ]:
result = chat.invoke("세계에서 가장 큰 산은?")

In [ ]:
result.content

In [ ]:
result

### (3) 메시지 구성
* 객체 방식

In [ ]:
# 동일한 코드
result1 = chat.invoke("세계에서 가장 큰 산은?")
result2 = chat.invoke([HumanMessage("세계에서 가장 큰 산은?")])

In [ ]:
# 역할부여
s_msg = "너는 애국심을 가지고 있는 건전한 대한민국 국민이야."
h_msg = "독도는 어느나라 땅이야?"

result = chat.invoke([SystemMessage(s_msg), HumanMessage(h_msg)])
result.content

### (4) 실습🔥

* 사용 모델: gpt-4.1-mini
* 사용자 질문: "생성형 AI를 처음 배우는 사람에게 LangChain을 3문장으로 설명해줘."
* 아래 2가지 시스템 메시지를 각각 적용하여 답변을 비교하세요.
    * 시스템 메시지 A : "너는 나의 절친이야. 쉬운 단어로 설명해."
    * 시스템 메시지 B : "너는 실무 중심의 AI 컨설턴트야. 핵심만 간결하게 설명해."


In [ ]:
# 1. 모델 준비


# 2. 사용자 질문


# 3. 시스템 메시지 A


## 실행


## 실행결과 확인


# 4. 시스템 메시지 B


## 실행


## 실행결과 확인


### (5) 답변의 다양성 조절

In [ ]:
question = "학생들의 학습 의욕을 높일 수 있는 짧은 응원 문구를 3개 작성해줘."

# temperature = 0
llm_low = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

# temperature = 1
llm_high = ChatOpenAI(model_name="gpt-4.1-mini", temperature=1)

response_low = llm_low.invoke(question)
response_high = llm_high.invoke(question)

print("[temperature = 0]")
print(response_low.content)
print("=" * 100)

print("[temperature = 1]")
print(response_high.content)

## **3. 프롬프트 구성**

### (1) 모델 선언

In [ ]:
chat = ChatOpenAI(model = "gpt-4.1-mini")

### (2) 프롬프트 구성하기
* System Message
    * 역할
    * 지침
    * 출력형식
* Human Message
    * 요청사항

In [ ]:
sys_msg = '''
# 역할
너는 입력하는 도시와 여행 기간을 입력받아서 여행 계획을 수립하는 여행 플래너야

# 지침
- 도시에서 유명한 관광지를 먼저 찾아
- 여행기간을 감안하여 일정을 수립해

# 출력형식
- bullet point 형식
- 일정별 오전, 오후로 나눠서 간결하게 설명
'''

human_msg = '''
# 여행 도시 : 부산
# 여행 기간 : 2박3일
'''

### (3) LLM 호출

In [ ]:
result = chat.invoke([SystemMessage(sys_msg), HumanMessage(human_msg)])
print(result.content)

### (4) 실습🔥

* 상품을 입력하면, 상품 홍보용 마케팅 문구 3가지를 제시하도록 메시지를 구성해 봅시다.
* 시스템 메시지 구성시 지침과 출력 형식을 구체적으로 작성해 봅시다.

## **4. 출력형식 다루기**

### (1) 실습🔥

* 3-(3)의 결과를 다음과 같이 출력하도록 프롬프트를 수정해 봅시다.

        {
        '1일차': {
            '오전': ['해운대 해수욕장', '동백섬'],
            '오후': ['광안리 해수욕장', '광안대교 야경 감상']
        },
        '2일차': {
            '오전': ['자갈치 시장', '부산 타워'],
            '오후': ['감천문화마을', '부산 영화의 전당']
        },
        '3일차': {
            '오전': ['태종대 공원'],
            '오후': ['국제시장', '부산 근대 역사관']
        }
        }

In [ ]:
sys_msg = '''
# 역할


# 지침


# 출력형식

'''

human_msg = '''
# 여행 도시 : 부산
# 여행 기간 : 2박3일
'''

In [ ]:
result = chat.invoke([SystemMessage(sys_msg), HumanMessage(human_msg)])
print(result.content)

### (2) llm의 출력 형식

In [ ]:
print(type(result), type(result.content))

In [ ]:
dict1 = result.content
dict1

### (3) ast.literal_eval()
문자열 형태의 Python dict/list를 실제 객체로 변환


In [ ]:
import ast
dict2 = ast.literal_eval(result.content)

In [ ]:
print(type(dict2))
print(dict2)

### (4) 실습🔥

* 장르를 입력하면, 추천영화 1편에 대해,
    * 영화 제목, 감독, 주연배우, 연도를 출력